In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "pypdf", "-q"])

import re
import os
import pandas as pd
import smtplib
import time
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from pypdf import PdfReader

SENDER_EMAIL = "amansaini.data.acc@gmail.com"
APP_PASSWORD = "cgsq fdgi stuy vltd"
PDF_FILE_NAME = r"C:\Users\Dell Laptop\Downloads\ACC_Gmail_List.pdf.pdf"

def extract_data_from_pdf(pdf_path):
    candidates = []
    if not os.path.exists(pdf_path):
        return []

    try:
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"

        for line in text.split('\n'):
            line = line.strip()
            if not line:
                continue

            email_match = re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', line)
            if not email_match:
                continue

            email = email_match.group(0).strip().lower()

            if "gmail address" in line.lower():
                continue

            name_part = line.replace(email_match.group(0), '').strip()
            name_part = re.sub(r'^\d+\s*', '', name_part)
            name_part = re.sub(r'["\',]', '', name_part).strip()

            if not name_part or len(name_part) < 2:
                name_part = email.split('@')[0].replace('.', ' ').title()

            candidates.append({"Name": name_part, "Email": email})

        return candidates
    except Exception:
        return []

def get_email_body(candidate_name):
    return f"""
    <html>
      <body>
        <p>Dear {candidate_name},</p>
        <p>We are thrilled to officially offer you an internship position as a <b>Data Analyst Intern</b> with us at ACC Analytical Career Connect!</p>
        <p>As a growing startup, ACC is a place where innovation and hustle thrive. We pride ourselves on providing a full corporate environment where our interns don't just observe—they actively contribute. During your time with us, you can expect a steep learning curve, hands-on experience with real-world projects, and the opportunity to make a direct impact on our operations.</p>
        <p>As outlined in the <b>Job Description (JD)</b> we shared previously, this role requires dedication and a strong willingness to learn. You can review the full details of your responsibilities and our expectations here:<br>
        <a href="https://docs.google.com/document/d/1R4ntZK5fyPuaernxjad7Kn8h88xWaXXCFALKgKY4kVg/edit?tab=t.6zw2w3881ylc">View Job Description</a></p>
        <p>We believe this internship will be a fantastic stepping stone for your career, offering you the mentorship, practical skills, and professional exposure you need to succeed in the corporate world.</p>
        <p>If you are excited to join our journey and are fully comfortable with the roles outlined in the JD, <b>please reply to this email to confirm your acceptance of this offer.</b> Once we receive your confirmation, we will share the final details regarding your joining date and the onboarding process.</p>
        <p>Congratulations, and we look forward to having you on the team!</p>
        <p>Best regards,<br><br>
        <b>Aman Saini</b><br>
        HR Team | ACC Analytical Career Connect<br>
        LinkedIn: <a href="https://www.linkedin.com/in/aman-saini-912850372/">Connect with me</a></p>
      </body>
    </html>
    """

extracted_data = extract_data_from_pdf(PDF_FILE_NAME)

if extracted_data:
    df = pd.DataFrame(extracted_data)
    df['Email'] = df['Email'].str.strip().str.lower()
    df = df.drop_duplicates(subset=['Email'], keep='first')
    df = df[df['Email'].str.contains('@')]

    server = None
    try:
        server = smtplib.SMTP_SSL("smtp.gmail.com", 465)
        server.login(SENDER_EMAIL, APP_PASSWORD)

        for index, row in df.iterrows():
            name = str(row['Name']).strip()
            recipient_email = str(row['Email']).strip()

            try:
                msg = MIMEMultipart()
                msg['From'] = SENDER_EMAIL
                msg['To'] = recipient_email
                msg['Subject'] = "Internship Offer: Welcome to ACC Analytical Career Connect!"
                msg.attach(MIMEText(get_email_body(name), 'html'))

                server.send_message(msg)
                print(f"Sent: {recipient_email}")
            except Exception:
                pass
            time.sleep(3)

    except Exception:
        pass
    finally:
        if server:
            try:
                server.quit()
            except:
                pass